# Ingestão de dados para RAG com LLM

Este notebook tem como objetivo realizar o processamento e divisão de arquivos Markdown, organizando os textos extraídos para ingestão em uma vector store. O propósito final é habilitar a utilização desses dados em um agente RAG (Retrieval-Augmented Generation), permitindo consultas eficientes e contextualizadas. 

A abordagem inclui:

- **Configuração do modelo LLM**: Utilizando o modelo `deepseek-r1:8b` para interações e processamento de linguagem natural.
- **Ingestão de dados**: Extração e organização dos textos Markdown para armazenamento estruturado.
- **Integração com vector store**: Preparação dos dados para consultas otimizadas em um agente RAG.

Este fluxo garante que os textos sejam processados de forma eficiente, permitindo a criação de agentes inteligentes baseados em recuperação de informações.

## 0. Configuração do ambiente

In [ ]:
# Configurations

OLLAMA_LLM_MODEL = "deepseek-r1:8b"

OLLAMA_EMBEDDING_MODEL = "mxbai-embed-large:latest"
# OLLAMA_EMBEDDING_MODEL = "nomic-embed-text:latest"

BIBLE_MD_PATH = "../../data/processed/biblia/pdf_to_markdown_using_marker-pdf/Bíblia Sagrada O Antigo e Novo Testamento - 4 volumes - Vulgata Latina por Pe. Matos Soares 1927-1950.md"

## 1. Configurando Ollama

In [ ]:
import ollama

installed_models = list(map(lambda model: model.model, ollama.list().models))
installed_models

In [ ]:
if OLLAMA_LLM_MODEL not in installed_models:
    print(f'Model "{OLLAMA_LLM_MODEL}" is not installed. Pulling...')
    ollama.pull(OLLAMA_LLM_MODEL)
else:
    print(f'Model "{OLLAMA_LLM_MODEL}" is already installed.')

if OLLAMA_EMBEDDING_MODEL not in installed_models:
    print(f'Embedding model "{OLLAMA_EMBEDDING_MODEL}" is not installed. Pulling...')
    ollama.pull(OLLAMA_EMBEDDING_MODEL)
else:
    print(f'Embedding model "{OLLAMA_EMBEDDING_MODEL}" is already installed.')

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from pprint import pprint

llm = ChatOllama(
    model=OLLAMA_LLM_MODEL,
    temperature=0.8,
    # num_predict = 256,
    # other params ...
)

messages = [
    SystemMessage(
        """
        Você é o Ollama. 
        Você é um assistente de IA que fala português fluentemente.
        Você é amigável e útil. Você não deve se comportar como um humano, mas sim como uma IA.
        Você deve ser sempre educado e respeitoso.
        Você deve ser muito bem humorado.
        """
    ),
    HumanMessage("Ollá Ollama! Prove que você é tão bom quanto dizem que você é."),
]

pprint(llm.invoke(messages).content)

## 2. Extraindo texto dos arquivos Markdown

### 2.1. (Apenas demonstração) Extraindo texto dos arquivos Markdown

**Não vamos utilizar esse método.**

Motivo: ao invés, vamos extrair o markdown inteiro (com as marcacoes) e depois fazer o split em cada arquivo markdown. Assim, conseguimos manter a formatação original do arquivo markdown.

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document

loader = UnstructuredMarkdownLoader(BIBLE_MD_PATH)

data = loader.load()
assert len(data) == 1
assert isinstance(data[0], Document)
bible_content = data[0].page_content
print(bible_content[:250])

In [ ]:
bible_content[:1000]

### 2.2. Extraindo texto dos arquivos Markdown (formatado como markdown)

In [ ]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader(BIBLE_MD_PATH, encoding="utf-8")

bible_docs = loader.load()

bible_text = bible_docs[0].page_content

print(bible_text[11000:12000])

## 3. Dividindo o texto em partes menores

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("####", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(bible_text)

pprint(md_header_splits[600].metadata)

print("\n\n")
pprint(md_header_splits[600].page_content)

## 4. Criando embeddings para os textos

In [ ]:
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)

# Testing the embedding model
ollama_embeddings.embed_query("Hello, world!")[:10]

In [ ]:
# Não vamos utilizar
# Motivo: será utilizado internamente na vector store

# from tqdm import tqdm

# embeddings = []
# for split in tqdm(md_header_splits, desc="Generating embeddings"):
#     embeddings.append(ollama_embeddings.embed_query(split.page_content))
    
# embeddings[0][:10]


## 5. Criando a vector store

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(
    embedding=ollama_embeddings
)

_ = vector_store.add_documents(
    md_header_splits
)

## 6. Criando o agente RAG

In [ ]:
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")

example_messages = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

assert len(example_messages) == 1
print(example_messages[0].content)